# Notebook 05: Predicting Healthcare Expenses – Traditional vs. ML Regression

In [ ]:
# Specify the disk drive path to the folder containing the Synthea CSV files
# You'll change the path on the next line to match your system setup
my_path = "/Users/spb24/Downloads/coherent-11-07-2022/csv/"

In [ ]:
import pandas
import sqlite3
from datetime import datetime

In [ ]:
# Load data
patients = pandas.read_csv(my_path + "patients.csv")

In [ ]:
# Compute age from BIRTHDATE
patients['BIRTHDATE'] = pandas.to_datetime(patients['BIRTHDATE'], errors='coerce')
patients['age'] = (datetime.today() - patients['BIRTHDATE']).dt.days // 365

In [ ]:
# Clean and select relevant columns
model_df = patients[['age', 'GENDER', 'HEALTHCARE_EXPENSES']].dropna()

In [ ]:
# Create binary variable for gender: 1 if male, 0 if female
model_df['male'] = model_df['GENDER'].map({'M': 1, 'F': 0})

## Traditional Ordinary Least Squares Regression

In [ ]:
# Traditional Regression (OLS with statsmodels)
import statsmodels.api as sm

In [ ]:
X = model_df[['age', 'male']]
X = sm.add_constant(X)
y = model_df['HEALTHCARE_EXPENSES']

In [ ]:
OLS_model = sm.OLS(y, X).fit()
print(OLS_model.summary())

## Machine Learning Regression

In [ ]:
# Machine Learning Regression (with scikit-learn)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
X_ml = model_df[['age', 'male']]
y_ml = model_df['HEALTHCARE_EXPENSES']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_ml, y_ml, test_size=0.3, random_state=42)

In [ ]:
ml_model = LinearRegression()
ml_model.fit(X_train, y_train)

In [ ]:
y_pred = ml_model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R^2:", r2_score(y_test, y_pred))

## Explainability

In [ ]:
# -------------------------------
# EXPLAINABILITY COMPARISON
# -------------------------------
print("\n--- Traditional OLS Coefficients ---")
print(OLS_model.params)

### Interpreting the OLS Coefficients

- **const**: The baseline healthcare expense when both `age` and `male` are 0. (Not meaningful on its own — think of it as a starting offset.)
- **age**: For every additional year of age, expected healthcare expenses **increase by about \$8,733**.
- **male**: Being male **decreases** expected healthcare expenses by about **\$4,063**, compared to being female.

👉 Reminder: These are **average effects** — OLS fits a single line through the entire dataset, assuming straight, consistent relationships.


In [ ]:
print("\n--- ML Model Coefficients ---")
print(dict(zip(X_ml.columns, ml_model.coef_)))

### 📈 Comparing ML Model Coefficients

Notice that the ML model's coefficients for `age` and `male` are **very close** to the OLS model's coefficients.

- Why? Because the data is fairly simple, and both models are solving for the same basic relationship: predict healthcare expenses from age and gender.
- Small differences are due to the way scikit-learn and statsmodels set up the features (e.g., inclusion/exclusion of intercepts).

👉 In messy real-world data, machine learning models can start to diverge from classic linear regression!


## Reflect:
- Which model helps you better understand *how* age and gender relate to expenses?
- What would you need in order to explain the ML model's prediction for one specific patient?
- In more complex models (e.g., Random Forest, XGBoost), how do we explain the 'why' of a prediction?
- What are the risks of using a model you can't explain?
- How might explainability needs differ for a data scientist, a clinician, and a patient?


In [ ]:
# -------------------------------
# LIME EXPLANATION (local)
# -------------------------------
import lime
import lime.lime_tabular
import numpy as np

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    mode='regression'
)

In [ ]:
i = 0
exp = explainer.explain_instance(
    data_row=X_test.iloc[i],
    predict_fn=ml_model.predict
)

In [ ]:
exp.as_list()

### What LIME Just Did

- LIME built a **tiny, simple model** around one patient to explain the prediction.
- It asked: "In the neighborhood around this patient, what features matter most?"

👉 Even in simple models, LIME focuses **locally** — meaning the explanation can differ from the overall model trend!
